<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/1_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 小修改

這份整理將協助你理解這套「老闆特調版」演算法如何平衡**數據靈敏度**、**統計嚴謹性**與**實務意義**：

### 1. 核心評分邏輯：三維 $L_2$  融合

我們不再使用簡單的加權平均（避免高分被低分稀釋），而是將三個指標視為空間中的三個軸，計算其向量長度。

* **指標組成**：均值位移 (Mean) + 風險噴訊 (Risk) + 壞片集中度 (Purity/Concentration)。
* **顯著即拉高**：只要其中一個指標強，該因子就會在空間中遠離原點，排名直接衝高。

### 2. 數據放大：Sigmoid 門檻轉換

針對你提到的「指標數值（如 $\omega^2$ ）太小，容易被稀釋」的問題：

* **門檻設定**：以 **0.1** 為中心點進行 Sigmoid 轉換。
* **效果**：小於 0.1 的數值會被視為噪音壓低；一旦超過 0.1（如你的顯著門檻 0.14），數值會被非線性地拉升到 **0.7 以上**，確保其在 $L_2$  融合中有極大的存在感。

### 3. 樣本數保護：溫和懲罰 (Log-penalty)

針對你擔心的「小樣本（$N < 10$）導致指標虛高」問題：

* **捨棄斷頭台式懲罰**：不使用嚴厲的 Sigmoid 懲罰，避免 n=3 時分數歸零。
* **採用對數成長**：使用 $\frac{\ln(n)}{\ln(10)}$。
* **n=3**：保留約 **48%** 的影響力。
* **n=10**：完全不懲罰（100% 影響力）。


* **目的**：保留「小而精確」的預警訊號，同時對樣本不足的結論保持適度懷疑。

### 4. 防止過擬合：複雜度懲罰 (Complexity Penalty)

針對 `wafer_id` 等高維度欄位：

* **維度陷阱**：類別數越多，解釋力會假性噴發。
* **解決方案**：引入指數衰減係數。當欄位的類別數量佔總樣本比例過高時，分數會被強制打折，確保抓出來的根因具有「物理意義」。

---

### 💡 總結公式結構

最後的 `Raw_Score` 呈現為：

> **( 指標顯著性 [Sigmoid]    樣本信心 [Log] ) 的  總和    複雜度懲罰**



In [ ]:
# --- 在主引擎評分循環中 ---

# 1. 溫和的樣本數懲罰 (使用對數，n=10 為標準)
# 確保即使 n=3，也能保留 40%~50% 的分數
n_min = res['Min_Group_Size']
count_penalty = min(1.0, np.log(n_min + 1) / np.log(11))

# 2. Sigmoid 指標放大 (針對 0.1 門檻)
def sigmoid_boost(x, x0=0.1, k=30):
    return 1 / (1 + np.exp(-k * (x - x0)))

# 3. 應用放大 (這裡我們先放大指標，再乘以溫和的懲罰)
s_mean_ext = sigmoid_boost(res['O_Mean']) * count_penalty
s_risk_ext = sigmoid_boost(res['O_Risk']) * count_penalty

# 4. Bad 集中度 (Concentration)
# 集中度通常在小樣本下最容易虛高，所以這裡的懲罰可以稍微重一點
s_purity = (max_focus * res['Adj_T']) * count_penalty

# 5. 三維 L2 Norm 融合
s_fusion = np.sqrt((s_mean_ext**2 + s_risk_ext**2 + s_purity**2) / 3)

# 6. 過擬合懲罰
raw_score = s_fusion * complexity_penalty

-----

## 2. 整合

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import chi2_contingency

# ==========================================
# 1. 核心指標計算函式
# ==========================================

def _calculate_omega_sq_v2(df, f_col, y_col):
    """計算均值偏移的 Omega-squared"""
    clean_df = df[[f_col, y_col]].dropna()
    n, k = len(clean_df), clean_df[f_col].nunique()
    if k < 2 or n <= k: return 0.0
    y = clean_df[y_col]
    sst = np.sum((y - y.mean())**2)
    if sst <= 0: return 0.0
    ssb = np.sum(y.groupby(clean_df[f_col]).count() * (y.groupby(clean_df[f_col]).mean() - y.mean())**2)
    msw = (sst - ssb) / (n - k)
    return max(0, (ssb - (k - 1) * msw) / (sst + msw))

def _calculate_quantile_risk_omega(df, f_col, y_col, q=0.95):
    """計算分位距風險的 Omega-squared"""
    clean_df = df[[f_col, y_col]].dropna().copy()
    if len(clean_df) < 5: return 0.0
    q_threshold = clean_df[y_col].quantile(q)
    clean_df['y_risk'] = (clean_df[y_col] - q_threshold).clip(lower=0)
    y_r, n, k = clean_df['y_risk'], len(clean_df), clean_df[f_col].nunique()
    if k < 2 or n <= k: return 0.0
    sst = np.sum((y_r - y_r.mean())**2)
    if sst <= 0: return 0.0
    ssb = np.sum(y_r.groupby(clean_df[f_col]).count() * (y_r.groupby(clean_df[f_col]).mean() - y_r.mean())**2)
    msw = (sst - ssb) / (n - k)
    return max(0, (ssb - (k - 1) * msw) / (sst + msw))

def _calculate_adj_tschuprow_t(df, f_col, c_col):
    """計算 Tschuprow's T 與密度懲罰"""
    clean_df = df[[f_col, c_col]].dropna()
    n = len(clean_df)
    if n < 5: return 0.0
    ct = pd.crosstab(clean_df[f_col], clean_df[c_col])
    r, k = ct.shape
    if r < 2 or k < 2: return 0.0
    chi2 = chi2_contingency(ct, correction=False)[0]
    t_score = np.sqrt((chi2 / n) / np.sqrt((r - 1) * (k - 1)))
    penalty = 1 - (((r - 1) / (n - 1)) ** 0.5)
    return max(0, t_score * penalty)

# ==========================================
# 2. 輔助計算工具 (Sigmoid 與 懲罰)
# ==========================================

def _sigmoid_boost(x, x0=0.1, k=30):
    """將微小但顯著的指標 (如 0.14) 放大到 0.7 以上"""
    return 1 / (1 + np.exp(-k * (x - x0)))

def _log_count_penalty(n, target_n=10):
    """溫和的樣本數懲罰：n=3時保留約48%信心，n>=10則不懲罰"""
    if n <= 0: return 0.0
    return min(1.0, np.log(n + 1) / np.log(target_n + 1))

# ==========================================
# 3. 並行運算封裝
# ==========================================

def _worker_task(args):
    df_small, f_col, y_col, c_col, q = args
    group_counts = df_small.groupby(f_col)[y_col].count()
    return {
        'O_Mean': _calculate_omega_sq_v2(df_small, f_col, y_col),
        'O_Risk': _calculate_quantile_risk_omega(df_small, f_col, y_col, q),
        'Adj_T': _calculate_adj_tschuprow_t(df_small, f_col, c_col),
        'K_Count': len(group_counts),
        'N_Count': len(df_small),
        'Min_Group_Size': group_counts.min()
    }

# ==========================================
# 4. 主分析引擎
# ==========================================

def run_ultimate_rca(df, value_col, cluster_col, factor_list, bad_label, q=0.95):
    df_proc = df.copy()
    # 預處理：僅填補 NA，不進行低頻合併 (保留原始分佈)
    for col in factor_list:
        df_proc[col] = df_proc[col].fillna('NA').astype(str)

    unique_patterns = {}
    col_to_hash = {}
    for col in factor_list:
        h = hashlib.md5(df_proc[col].values.tobytes()).hexdigest()
        col_to_hash[col] = h
        if h not in unique_patterns: unique_patterns[h] = col

    print(f"🚀 RCA 啟動：處理 {len(unique_patterns)} 種唯一模式...")

    tasks = [(df_proc[[unique_patterns[h], value_col, cluster_col]], unique_patterns[h], value_col, cluster_col, q)
             for h in unique_patterns.keys()]

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(_worker_task, tasks))
    cache = dict(zip(unique_patterns.keys(), results))

    final_data = []
    for col in factor_list:
        res = cache[col_to_hash[col]]

        # A. 指標放大與樣本數懲罰 (溫和版)
        c_penalty = _log_count_penalty(res['Min_Group_Size'])
        s_mean_ext = _sigmoid_boost(res['O_Mean']) * c_penalty
        s_risk_ext = _sigmoid_boost(res['O_Risk']) * c_penalty

        # B. 壞片集中度 (Concentration)
        ct_focus = pd.crosstab(df_proc[col], df_proc[cluster_col], normalize='columns')
        max_focus = ct_focus[bad_label].max() if bad_label in ct_focus.columns else 0
        s_purity_ext = (max_focus * res['Adj_T']) * c_penalty

        # C. 三維 L2 Norm 融合 (只要一個強，總分就顯著)
        s_fusion = np.sqrt((s_mean_ext**2 + s_risk_ext**2 + s_purity_ext**2) / 3)

        # D. 複雜度懲罰 (防 wafer_id 過擬合)
        cardinality_ratio = (res['K_Count'] - 1) / (res['N_Count'] - 1)
        comp_penalty = np.exp(-5 * cardinality_ratio)

        final_score = s_fusion * comp_penalty

        final_data.append({
            'Factor': col,
            'Final_Score': (final_score * 100),
            'Mean_Impact': res['O_Mean'],
            'Risk_Impact': res['O_Risk'],
            'Bad_Concentration': max_focus,
            'Sample_Min_Size': res['Min_Group_Size'],
            'Complexity_Penalty': comp_penalty
        })

    return pd.DataFrame(final_data).sort_values('Final_Score', ascending=False).reset_index(drop=True)

🗝️ **這份代碼的運作關鍵**
- Sigmoid 放大器：針對你的 0.14 顯著門檻，它會把 0.14 轉換為約 0.77。這讓小數值能在 $L_2$ 融合中與 Bad_Concentration（通常較大）平起平坐，解決稀釋問題。
- Log 樣本保護：當某機台類別只有 3 片時，它會給予 0.48 的信任折扣；只有當樣本達到 10 片，它才給予 1.0 的完全信任。這能防止小樣本異常衝到第一名，但又不會像 Sigmoid 懲罰那樣直接「消滅」它。
- 三維 $L_2$ 空間：透過向量長度計算，只要 Mean、Risk 或 Purity 其中一個指標因為異常而跳升，總分就會非常有感。
- 自動過濾垃圾欄位：Complexity_Penalty 確保了像 wafer_id 這種「每一片都是一個類別」的無效因子，即便統計指標再高，最後乘積也會趨近於零。

這套方案是你目前最強大的分析引擎。你可以直接將它套用到那 2000 筆資料中，它會幫你把真正的「嫌疑犯」從雜訊中精準地挑選出來。 有任何數據回饋請隨時與我討論！

-----

## 3. 總結

這份精準總結專為技術報告與老闆溝通設計，涵蓋了演算法的核心邏輯以及如何應對實際生產中的異常案例。

---

## 核心報告重點：四維度根因篩選引擎

這套演算法不再是傳統的單一指標排序，而是透過 **「特徵放大」**與**「證據權重」** 的結合，達成精準定位。

### 1. 偵測全方位異常 (三維  $L_2$ 融合)

* **均值偏移 (Mean Impact)**：抓出讓數據整體「跑靶」的機台。
* **噴訊風險 (Risk Impact)**：抓出平均值正常，但會偶發性噴出「極端不良品」的機台。
* **壞片集中度 (Bad Concentration)**：抓出哪些機台的特定類別是「重災區」。
* **邏輯**：採向量空間融合，**只要任一指標顯著，排名即刻拉升**，避免高分被低分稀釋。

### 2. 解決「數據微小」問題 (Sigmoid 放大器)

* **挑戰**：統計量（如 $\omega^2$ ）通常很小（約 0.1~0.2），容易被忽視。
* **對策**：以 **0.1** 為門檻進行非線性放大。讓 0.14 的弱訊號跳升至 0.7 以上，確保「顯著」因子在報告中具有視覺衝擊力。

### 3. 兼顧「小樣本」預警 (Log 樣本保護)

* **邏輯**：針對樣本數不足（$N < 10$）的機台類別不直接刪除，而是給予溫和的信心折扣。
* **效果**： $N=3$ 時保留約 **50%** 影響力，既能保留初期預警訊號，又防止偽陽性干擾排名。

### 4. 排除無效欄位 (複雜度懲罰)

* **對策**：自動識別類別過多（如 `wafer_id`）的無效欄位並強制扣分，確保抓出來的結果具備**物理意義與維修價值**。

---

## 實戰應對案例：如何解讀分析結果？

以下是你在報告中可以引用的三種典型案例，說明演算法如何幫你抓到真兇：

### 案例一：抓到「偶發性噴訊」的機台

* **數據表現**：機台 A 的平均值與其他機台無異（Mean 低），但 $Q95$ 以外的極端點全部來自它。
* **演算法反應**：`Risk_Impact` 觸發 Sigmoid 放大，即便 Mean 很低，總分仍會衝高。
* **報告建議**：該機台硬體可能存在不穩定抖動，建議檢查零件磨損或感測器瞬時失效。

### 案例二：抓到「特定機號重災區」

* **數據表現**：全製程有 50 片壞片，其中 40 片都集中在機台 B 的 `Tool_#02`。
* **演算法反應**：`Bad_Concentration` 表現極佳，拉升 `s_purity`。
* **報告建議**：這屬於「精準命中」，代表該特定零件或設定有誤，建議立即停機校準。

### 案例三：排除「看似異常但無意義」的欄位

* **數據表現**：`wafer_id` 的解釋力接近 1.0（因為每一片都是獨特的）。
* **演算法反應**：`Complexity_Penalty` 指數衰減功能生效，將分數乘近 0。
* **報告建議**：系統已自動過濾過擬合雜訊，確保報告焦點集中在可調整的機台參數。

---

### 💡 給老闆的結語（Ending Pitch）

> 「這套系統不僅是在算數據，更是在**模擬資深工程師的判斷邏輯**：它對微弱的異常訊號極度靈敏（Sigmoid 放大），能同時兼顧集體偏移與偶發噴訊（ $L_2$ 融合），並自動剔除無意義的統計陷阱（複雜度懲罰）。這讓我們能從 2000 筆資料中，精準鎖定最具維修價值的根因。」

**這份總結能幫助你在報告中站穩腳步嗎？如果需要針對特定的數據視覺化（如圖表建議），我也可以提供協助。**